# Search

Many problems have the same shape: get from a **start state** to a **goal state** through a sequence of actions. A route from one town to another, the moves that solve a puzzle, the steps of a plan. The states and actions form a **graph**: each state is a node, and each action an edge to the state it leads to.

A search algorithm explores that graph from the start until it reaches the goal. The algorithms below differ only in **which node they explore next**, and that choice decides whether the path they return is the shortest, the cheapest, or just a path.

## Breadth-first search

**Breadth-first search (BFS)** explores the graph by levels: first every node one step from the start, then every node two steps away, and so on. It keeps the nodes waiting to be explored, the **frontier**, in a first-in, first-out queue. Because it reaches every node at the lowest possible number of steps, the first path it finds to the goal has the **fewest steps**.

A small road network, as a dictionary from each town to its neighbours:

In [ ]:
road_network = {
    "A": ["C", "B"],
    "B": ["A", "D", "E"],
    "C": ["A", "F"],
    "D": ["B"],
    "E": ["B", "F", "G"],
    "F": ["C", "E"],
    "G": ["E"],
}

Each entry in the queue is a path from the start; the last town of the path is the node to explore. `visited` stops the search from queuing a town twice.

In [ ]:
from collections import deque


def breadth_first_search(graph, start, goal):
    frontier = deque([[start]])
    visited = {start}
    explored = []
    while frontier:
        path = frontier.popleft()
        node = path[-1]
        explored.append(node)
        if node == goal:
            return path, explored
        for neighbour in graph[node]:
            if neighbour not in visited:
                visited.add(neighbour)
                frontier.append(path + [neighbour])
    return None, explored


path, explored = breadth_first_search(road_network, "A", "G")
print("Explored:", " ".join(explored))
print("Path:    ", " → ".join(path))

Explored: A C B F D E G
Path:     A → B → E → G


## Depth-first search

**Depth-first search (DFS)** follows one branch as deep as it goes and only backs up at a dead end: its frontier is a last-in, first-out stack, here the call stack of a recursive function. Without a `visited` set it needs memory only for the current path, which is its advantage in huge state spaces; with one, as here, it can't loop forever in a graph with cycles.

It finds *a* path, not necessarily the shortest: it returns the first one it stumbles on.

In [ ]:
def depth_first_search(graph, node, goal, path=None, visited=None):
    path = (path or []) + [node]
    visited = visited if visited is not None else set()
    visited.add(node)
    if node == goal:
        return path
    for neighbour in graph[node]:
        if neighbour not in visited:
            found = depth_first_search(graph, neighbour, goal, path, visited)
            if found:
                return found
    return None


print("Path:", " → ".join(depth_first_search(road_network, "A", "G")))

Path: A → C → F → E → G


DFS went through C first, because C comes first in A's list of neighbours, and reached the goal in four steps; BFS found a path of three.

## Uniform-cost search

Fewest steps is not cheapest when the steps have different costs, such as the kilometres of each road. **Uniform-cost search (UCS)**, the search version of Dijkstra's algorithm, always explores the frontier node with the **lowest cost so far**, written g(n). Its frontier is a priority queue, and the first time it takes the goal out of the queue, it has found the cheapest path.

A map where every town has coordinates, and every road is at least as long as the straight line between its two towns:

In [ ]:
town_positions = {
    "S": (0, 0), "A": (3, 1), "B": (6, 1), "G": (10, 0), "C": (-3, 0),
    "D": (-3, 4), "E": (0, 5), "F": (5, -3), "H": (8, -3),
}
roads = [
    ("S", "A", 4), ("A", "B", 3), ("B", "G", 5), ("S", "C", 3), ("C", "D", 4),
    ("S", "E", 5), ("E", "D", 4), ("S", "F", 6), ("F", "H", 3), ("H", "G", 4),
]

weighted_network = {town: [] for town in town_positions}
for town_a, town_b, length in roads:
    weighted_network[town_a].append((town_b, length))
    weighted_network[town_b].append((town_a, length))

Going from S to G costs 4 + 3 + 5 = 12 through A and B, and 6 + 3 + 4 = 13 through F and H. C, D and E lie behind the start, away from the goal.

In [ ]:
import heapq


def uniform_cost_search(graph, start, goal):
    frontier = [(0, start, [start])]
    expanded = []
    while frontier:
        cost, node, path = heapq.heappop(frontier)
        if node in expanded:
            continue
        expanded.append(node)
        if node == goal:
            return path, cost, expanded
        for neighbour, length in graph[node]:
            if neighbour not in expanded:
                heapq.heappush(frontier, (cost + length, neighbour, path + [neighbour]))
    return None, float("inf"), expanded


path, cost, expanded = uniform_cost_search(weighted_network, "S", "G")
print("Path:", " → ".join(path), "| cost", cost)
print("Expanded", len(expanded), "towns:", " ".join(expanded))

Path: S → A → B → G | cost 12
Expanded 9 towns: S C A E F B D H G


UCS found the cheapest path, but it expanded C, D and E as well: it grows a circle of cost around the start, whatever direction the goal is in.

## A* search

**A\*** adds an estimate of the cost still to go, the **heuristic** h(n), and explores the node with the lowest f(n) = g(n) + h(n): the cost so far plus the estimated cost to the goal. Here the heuristic is the straight-line distance to G, which the road positions give for free.

A\* returns the cheapest path if the heuristic never **overestimates** the real remaining cost (it is **admissible**). This version never re-expands a town, which needs a slightly stronger condition, **consistency**: along every road, the estimate can't drop by more than the road's length. A straight-line distance has both properties, because no road is shorter than the straight line.

In [ ]:
import math


def straight_line_to_goal(town):
    return math.dist(town_positions[town], town_positions["G"])


def a_star(graph, start, goal, heuristic):
    frontier = [(heuristic(start), 0, start, [start])]
    expanded = []
    while frontier:
        _, cost, node, path = heapq.heappop(frontier)
        if node in expanded:
            continue
        expanded.append(node)
        if node == goal:
            return path, cost, expanded
        for neighbour, length in graph[node]:
            if neighbour not in expanded:
                new_cost = cost + length
                heapq.heappush(frontier, (new_cost + heuristic(neighbour), new_cost, neighbour, path + [neighbour]))
    return None, float("inf"), expanded


path, cost, expanded = a_star(weighted_network, "S", "G", straight_line_to_goal)
print("Path:", " → ".join(path), "| cost", cost)
print("Expanded", len(expanded), "towns:", " ".join(expanded))

Path: S → A → B → G | cost 12
Expanded 5 towns: S A B F G


The same path and the same cost, with fewer towns expanded: the heuristic kept A\* away from C, D and E. With a heuristic that is always 0, A\* *is* uniform-cost search:

In [ ]:
path, cost, expanded = a_star(weighted_network, "S", "G", lambda town: 0)
print("Path:", " → ".join(path), "| cost", cost, "| expanded", len(expanded))

Path: S → A → B → G | cost 12 | expanded 9


An estimate that **overestimates** breaks the guarantee. Doubling the straight-line distance, a trick called *weighted A\**, makes the search greedier: it expands even fewer towns, and it returns a path that is not the cheapest.

In [ ]:
path, cost, expanded = a_star(weighted_network, "S", "G", lambda town: 2 * straight_line_to_goal(town))
print("Path:", " → ".join(path), "| cost", cost, "| expanded", len(expanded))

Path: S → F → H → G | cost 13 | expanded 4


Route planners and game engines make exactly this trade-off: a slightly worse path, found much sooner.

## The four algorithms side by side

In the usual notation, *b* is the branching factor (how many successors a node has), *d* the depth of the shallowest solution and *m* the maximum depth of the graph.

| Algorithm | Frontier | Path it returns | Memory |
| --- | --- | --- | --- |
| Breadth-first | Queue (FIFO) | Fewest steps | O(b^d): the whole level |
| Depth-first | Stack (LIFO) | Any path; in finite graphs it finds one if it exists | O(b·m) without a visited set |
| Uniform-cost | Priority queue on g(n) | Cheapest | Up to O(b^d), and more with cheap steps |
| A\* | Priority queue on g(n) + h(n) | Cheapest, if h is admissible (and consistent, without re-expansion) | Like UCS in the worst case; much less with a good heuristic |

The time each one takes grows like the number of nodes it explores, which is why the heuristic matters: a good one cuts the exploration down to little more than the path itself.

## Searching against an opponent

In a game, the other player moves too, so a plan must work **whatever the opponent does**. **Minimax** searches the game tree assuming both players play their best: on its own turns it picks the move with the highest value, on the opponent's turns the move with the lowest. Tic-tac-toe is small enough (5,478 possible positions) to search to the end, so a minimax player never loses.

[Python tic-tac-toe](https://github.com/avidaldo/python-tic-tac-toe) builds one: its level 2 implements minimax, and [MINIMAX_ALGORITHM.md](https://github.com/avidaldo/python-tic-tac-toe/blob/master/02_minimax/MINIMAX_ALGORITHM.md) explains the algorithm step by step.

Bigger games need more:

- **Alpha-beta pruning** skips branches that can't change the decision, and gives the same move as minimax with far less search.
- **Depth-limited search** stops at a fixed depth and scores the positions there with an **evaluation function**, a heuristic, as in A\*. Chess engines did this for decades with hand-written evaluations.
- **AlphaGo** (2016) and **AlphaZero** (2017) kept the search, as Monte Carlo tree search, and replaced the hand-written evaluation with neural networks that learned it from games: classical search and machine learning in one system.